In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
df=pd.read_csv("/content/spam.csv",encoding='latin1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [8]:
df.drop(["Unnamed: 2","Unnamed: 3","Unnamed: 4"],axis=1,inplace=True)

In [9]:
df.columns

Index(['v1', 'v2'], dtype='object')

In [10]:
df.isna().sum()

,0
v1,0
v2,0


In [12]:
import nltk
import re


In [28]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [37]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
processed=[]

for text in df['v2']:
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens=[word for word in tokens if word not in stop_words]
    tokens=[lemmatizer.lemmatize(word) for word in tokens]

    processed.append(" ".join(tokens))



In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report

In [41]:
tf_idf=TfidfVectorizer()

new_x=tf_idf.fit_transform(processed)

In [44]:
print(new_x.toarray())

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [45]:
new_x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 44934 stored elements and shape (5572, 7019)>

In [46]:
y=df['v1']

In [47]:
new_x_train,new_x_test,y_train,y_test=train_test_split(new_x,y,test_size=0.2,random_state=42)

In [49]:
bernb=BernoulliNB()
b_model=bernb.fit(new_x_train,y_train)

In [50]:
b_model.score(new_x_test,y_test)

0.9713004484304932

In [53]:
from sklearn.metrics import confusion_matrix,classification_report
y_pred=b_model.predict(new_x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[958   7]
 [ 25 125]]
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       965
        spam       0.95      0.83      0.89       150

    accuracy                           0.97      1115
   macro avg       0.96      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [55]:
gaunb=GaussianNB()
g_model=gaunb.fit(new_x_train.toarray(),y_train)

In [57]:
g_model.score(new_x_test.toarray(),y_test)

0.8726457399103139

In [59]:
y_pred2=g_model.predict(new_x_test.toarray())
print(confusion_matrix(y_test,y_pred2))
print(classification_report(y_test,y_pred2))

[[847 118]
 [ 24 126]]
              precision    recall  f1-score   support

         ham       0.97      0.88      0.92       965
        spam       0.52      0.84      0.64       150

    accuracy                           0.87      1115
   macro avg       0.74      0.86      0.78      1115
weighted avg       0.91      0.87      0.88      1115



In [60]:
m_model=MultinomialNB()
m_model.fit(new_x_train,y_train)

MultinomialNB()

In [62]:
m_model.score(new_x_test,y_test)

0.9623318385650225

In [63]:
y_pred3=m_model.predict(new_x_test)
print(confusion_matrix(y_test,y_pred3))
print(classification_report(y_test,y_pred3))

[[965   0]
 [ 42 108]]
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       965
        spam       1.00      0.72      0.84       150

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115

